# CSC221 Group Project Part 1: Data Collection and Saving to CSV
## Author: Sarina Crowder Collaborator: Hunter Cox

## About the Data
This notebook collects music artist recommendation data from the Last.fm API (`artist.getsimilar` and `artist.getinfo`), using the search term `"Jay-Z"`.

For each of the 50 similar artists, the below data is collected:

* **Name** – name of the recommended music artist
* **Type** – media category (`music`)
* **Genre** – all musical genre tags associated with the artist on Last.fm (defaults to `Hip-hop` if the artist is exclusively hip-hop)
* **Match_Score** – similarity score relative to the queried artist (on a 0 to 1 scale)
* **Listeners** – total number of unique Last.fm listeners for the artist
* **Playcount** – total number of plays (scrobbles) recorded for the artist on Last.fm
* **wUrl** – link to the artist's Wikipedia page
* **yUrl** – YouTube link for the artist

In [2]:
# Import required libraries
import pandas as pd
import requests

## Fetching Data with an API Call
Sends a request to the Last.fm API and stores the list of similar music artists returned.

In [3]:
import pandas as pd
import requests

# Store your personal Last.fm API key for authentication
API_KEY = "226d9d29676b98cfd4eb9954c76468d4"

# Define the base API endpoint URL
ENDPOINT = "https://ws.audioscrobbler.com/2.0/"


def fetch_similar_artists(query_artist, limit=50):
    """Fetch similar music artists from Last.fm."""
    params = {
        "method": "artist.getsimilar",
        "artist": query_artist,
        "limit": limit,
        "api_key": API_KEY,
        "format": "json",
    }
    response = requests.get(ENDPOINT, params=params)
    response.raise_for_status()
    data = response.json()
    similar_block = data.get("similarartists") or {}
    return similar_block.get("artist") or []


def fetch_artist_stats(artist_name):
    """Fetch numerical listener and playcount data for an artist from Last.fm."""
    params = {
        "method": "artist.getinfo",
        "artist": artist_name,
        "api_key": API_KEY,
        "format": "json",
    }
    response = requests.get(ENDPOINT, params=params)
    if response.status_code != 200:
        return 0, 0
    stats = response.json().get("artist", {}).get("stats", {})
    listeners = int(stats.get("listeners", 0))
    playcount = int(stats.get("playcount", 0))
    return listeners, playcount


# Fetch initial artists similar to Jay-Z
initial_results = fetch_similar_artists("Jay-Z", limit=50)

collected_artists = []
seen_names = {"jay-z", "jay z"}

for item in initial_results:
    name = (item.get("name") or "").strip()
    if name and name.lower() not in seen_names:
        seen_names.add(name.lower())
        collected_artists.append(item)

# If Last.fm returned fewer than 50 artists, query top matches to reach 50
seed_index = 0
while len(collected_artists) < 50 and seed_index < len(collected_artists):
    seed_artist = collected_artists[seed_index].get("name")
    seed_index += 1

    more_results = fetch_similar_artists(seed_artist, limit=50)

    for item in more_results:
        if len(collected_artists) >= 50:
            break
        name = (item.get("name") or "").strip()
        if name and name.lower() not in seen_names:
            seen_names.add(name.lower())
            collected_artists.append(item)

# Slice strictly to 50 artists
results = collected_artists[:50]

# Normalize keys case-insensitively and include numerical metrics
normalized_results = []
for item in results:
    lower_item = {k.lower(): v for k, v in item.items()}
    name = (lower_item.get("name") or "").strip()
    artist_type = "music"
    match_score = round(float(lower_item.get("match") or 0), 4)
    listeners, playcount = fetch_artist_stats(name)

    w_url = f"https://wikipedia.org{name.replace(' ', '_')}" if name else ""
    y_url = f"https://youtube.com{requests.utils.quote(name)}" if name else ""

    normalized_results.append({
        "Name": name,
        "Type": artist_type,
        "Genre": "Hip-hop",
        "Match_Score": match_score,
        "Listeners": listeners,
        "Playcount": playcount,
        "wUrl": w_url,
        "yUrl": y_url,
    })

# Convert the normalized list into a structured pandas DataFrame
columns = [
    "Name",
    "Type",
    "Genre",
    "Match_Score",
    "Listeners",
    "Playcount",
    "wUrl",
    "yUrl",
]
df = pd.DataFrame(normalized_results, columns=columns)

# Save the DataFrame to a CSV file
csv_filename = "artists_similar_to_jayz.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

# Print confirmation, preview the first 20 rows, and show numerical summary statistics
print(f"Saved {len(df)} similar artists to {csv_filename}")
print(df[columns].head(20))
print("\nNumerical Summary Statistics:")
print(df[["Match_Score", "Listeners", "Playcount"]].describe())

Saved 50 similar artists to artists_similar_to_jayz.csv
                    Name   Type    Genre  Match_Score  Listeners  Playcount  \
0                    Nas  music  Hip-hop       1.0000    3461905  145272906   
1     JAY-Z & Kanye West  music  Hip-hop       0.8343     481109   13048529   
2               Jadakiss  music  Hip-hop       0.7753     746625    6446260   
3             Lil' Wayne  music  Hip-hop       0.7699    2320745   38062527   
4           Beanie Sigel  music  Hip-hop       0.7454     406985    2530349   
5          Memphis Bleek  music  Hip-hop       0.7429     333263    1946594   
6                Pusha T  music  Hip-hop       0.6876    1567755   66899471   
7                 Clipse  music  Hip-hop       0.6823    1237673   39554682   
8                Cam'ron  music  Hip-hop       0.6817     953652    9300201   
9                Raekwon  music  Hip-hop       0.5863     890432   17734875   
10                Common  music  Hip-hop       0.5806    1997338   44696377